In [ ]:
# SPDX-FileCopyrightText: Copyright 2025 Idiap Research Institute <contact@idiap.ch>
# SPDX-FileContributor: Anshul Gupta <anshul.gupta@idiap.ch>
# SPDX-License-Identifier: GPL-3.0-only

"""
Post-process gaze following prediction for lah
"""
import numpy as np
import torch
import pickle
import itertools
from tqdm import tqdm
import io
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu")
        else:
            return super().find_class(module, name)


def load_pkl(path_to_file):
    with open(path_to_file, "rb") as file:
        return CPU_Unpickler(file).load()


def to_numpy(tens):
    try:
        return tens.cpu().numpy()
    except:
        return tens.numpy()


def squeeze_res(res):
    new_res = {}
    for k, v in res.items():
        try:
            new_res[k] = v.squeeze(0)
        except AttributeError:
            new_res[k] = v
    return new_res


def is_inside(head_bbox, gaze_pt):
    if (
        gaze_pt[0] > head_bbox[0]
        and gaze_pt[0] < head_bbox[2]
        and gaze_pt[1] > head_bbox[1]
        and gaze_pt[1] < head_bbox[3]
    ):
        return True
    return False

In [ ]:
# Post-process LAH
def process_lah(res):
    res = squeeze_res(res)
    head_boxes = to_numpy(res["head_bboxes"])
    num_people = head_boxes.shape[0]
    gaze_preds = to_numpy(res["gp_pred"])
    inout = to_numpy(res["inout_gt"])

    # get lah gt
    lah_gts = to_numpy(res["lah_gt"])
    lah_gts_ = []
    lah_preds = []

    # get lah pred
    pair_indices = np.array(list(itertools.permutations(torch.arange(num_people), 2)))
    for p in range(num_people):
        if res["dataset"][0] == "gazefollow":
            io = 1
            gaze_pred = gaze_preds
            if p != num_people - 1:
                lah_preds.append(0)
                lah_gts_.append(-1)
                continue
        else:
            io = inout[p]
            gaze_pred = gaze_preds[p]
        if io == 1:
            lah_pred = 0
            lah_gt = -1

            valid_indices = np.where(
                (pair_indices[:, 1] == p).astype(np.int32)
                * (pair_indices[:, 0] != 0).astype(np.int32)
            )[0]
            if (lah_gts[valid_indices] != -1).sum() > 0:
                lah_idx = np.where(lah_gts[valid_indices] == 1)[0]
                if len(lah_idx) > 0:
                    lah_gt = 1
                    lah_idx = lah_idx[0]
                    pair = pair_indices[valid_indices[lah_idx]]
                    if is_inside(head_boxes[pair[0]], gaze_pred):
                        lah_pred = 1
                else:
                    lah_gt = 0
                    for hi, head_bbox in enumerate(head_boxes):
                        if hi == p:
                            continue
                        if is_inside(head_bbox, gaze_pred):
                            lah_pred = 1

            lah_preds.append(lah_pred)
            lah_gts_.append(lah_gt)
    return lah_gts_, lah_preds

In [ ]:
# Prcoess lah per dataset
def compute_p_r_lah_dataset(test_preds, dataset=None):
    if dataset == None:
        dataset_test_preds = test_preds
    else:
        dataset_test_preds = [dt for dt in test_preds if dt["dataset"][0] == dataset]
    print(f"Info: Len Preds {dataset}: {len(dataset_test_preds)}")

    all_lah_gts = []
    all_lah_preds_pp = []
    for res in tqdm(dataset_test_preds):
        # Post-Process lah
        lah_gts_, lah_preds_pp = process_lah(res)
        all_lah_gts.extend(lah_gts_)
        all_lah_preds_pp.extend(lah_preds_pp)

    print(f"Info: Len Combined Preds {dataset}: {len(all_lah_preds_pp)}")

    # Calculate precision and recall
    all_lah_gts = np.array(all_lah_gts)
    all_lah_preds_pp = np.array(all_lah_preds_pp)
    mask = all_lah_gts != -1
    print(all_lah_gts.shape, all_lah_preds_pp.shape, mask.shape)
    print("Post-processing:")
    precision = precision_score(all_lah_gts[mask], all_lah_preds_pp[mask])
    recall = recall_score(all_lah_gts[mask], all_lah_preds_pp[mask])
    f1 = f1_score(all_lah_gts[mask], all_lah_preds_pp[mask])
    print("Prec: ", precision)
    print("Recall: ", recall)
    print("F1: ", f1)

In [ ]:
# Process lah for all the datatses
path_to_results = "/home/jinwoongjung/MTGS/experiments/2026-05-26/VSGaze_transformer/test_predictions.p"  # Path to the test predictions pkl file
test_preds = load_pkl(path_to_results)

/tmp/ipykernel_3921371/1310678334.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return lambda b: torch.load(io.BytesIO(b), map_location="cpu")


In [ ]:
# # for dataset in datasets:
# # datasets = ['videoattentiontarget']
# datasets = ['coatt', 'childplay', 'laeo', 'videoattentiontarget']
# # datasets = [None]
# for dataset in datasets:
#     compute_p_r_lah_dataset(test_preds, dataset)

In [ ]:
datasets = [None]
for dataset in datasets:
    compute_p_r_lah_dataset(test_preds, dataset)

Info: Len Preds None: 14528


100%|██████████| 14528/14528 [00:04<00:00, 3163.74it/s]

Info: Len Combined Preds None: 29662
(29662,) (29662,) (29662,)
Post-processing:
Prec:  0.8759454503781802
Recall:  0.7937175493250259
F1:  0.8328067117018958
